# Mechanistic Interpretability of Transformer Models
**Author:** Trinadh Kolluboyina

This notebook walks through all three interpretability techniques implemented in this project:
1. **Attention Visualization** — heatmaps, token influence, head diversity
2. **Feature Attribution** — Integrated Gradients and LIME
3. **Probing Classifiers** — what does each layer encode?

---

## 0. Setup

In [ ]:
import sys, os
sys.path.append(os.path.abspath(".."))

import warnings
warnings.filterwarnings("ignore")

import torch
import numpy as np
import matplotlib.pyplot as plt

print(f"PyTorch version : {torch.__version__}")
print(f"Device          : {'cuda' if torch.cuda.is_available() else 'cpu'}")

---
## 1. Attention Visualization

We extract raw attention weight tensors from BERT and visualize how tokens attend to each other.

In [ ]:
from src.attention_visualization import AttentionExtractor, AttentionVisualizer

extractor = AttentionExtractor("bert-base-uncased")
visualizer = AttentionVisualizer(extractor)

In [ ]:
sample_text = "The cat sat on the mat near the warm fireplace."

tokens, attentions = extractor.get_attentions(sample_text)
print(f"Tokens          : {tokens}")
print(f"Attention shape : {attentions.shape}  (layers, heads, seq_len, seq_len)")

### 1a. Average Attention Heatmap (all layers & heads)

In [ ]:
visualizer.plot_attention_heatmap(sample_text)

### 1b. Layer-Specific Heatmap

In [ ]:
# Compare early vs late layers
for layer in [0, 5, 11]:
    print(f"\n--- Layer {layer} ---")
    visualizer.plot_attention_heatmap(sample_text, layer=layer)

### 1c. Token Influence Plot

Which tokens attend most to a specific target token?

In [ ]:
# Token index 2 = 'cat'
visualizer.plot_token_influence(sample_text, target_token_idx=2, layer=6)

### 1d. Head Diversity — All Attention Heads at Layer 0

Different heads specialize in different syntactic/semantic relationships.

In [ ]:
visualizer.plot_head_diversity(sample_text, layer=0)

---
## 2. Feature Attribution

We use two methods to score token importance for a **sentiment classification** model:
- **Integrated Gradients** — gradient-based, exact attribution
- **LIME** — model-agnostic, perturbation-based

In [ ]:
from src.feature_attribution import GradientAttribution, LIMETextExplainer

MODEL = "distilbert-base-uncased-finetuned-sst-2-english"
grad_attr  = GradientAttribution(MODEL)
lime_expl  = LIMETextExplainer(MODEL)

In [ ]:
sentences = [
    "The movie was absolutely brilliant and deeply moving.",
    "A dull, predictable, and thoroughly boring film.",
    "The acting was superb but the plot was weak.",
]

### 2a. Integrated Gradients Attribution

In [ ]:
for sent in sentences:
    print(f"\nText: {sent}")
    grad_attr.plot_attributions(sent, target_class=1)

### 2b. LIME Explanation

In [ ]:
for sent in sentences:
    print(f"\nText: {sent}")
    importance = lime_expl.plot_explanation(sent, num_samples=300, target_class=1, top_k=10)
    top = sorted(importance.items(), key=lambda x: abs(x[1]), reverse=True)[:5]
    print("  Top tokens:", top)

### 2c. Gradient vs LIME Comparison

Let's compare the two methods side-by-side on one sentence.

In [ ]:
sent = "The movie was absolutely brilliant and deeply moving."

# Gradient attribution
grad_tokens, grad_scores = grad_attr.get_token_attributions(sent, target_class=1)
grad_norm = grad_scores / (grad_scores.max() + 1e-8)

# LIME
lime_importance = lime_expl.explain(sent, num_samples=300, target_class=1)
words = sent.split()
lime_scores = np.array([lime_importance.get(w, 0.0) for w in words])
lime_norm = lime_scores / (np.abs(lime_scores).max() + 1e-8)

fig, axes = plt.subplots(2, 1, figsize=(12, 7))

# Gradient
word_tokens = [t for t in grad_tokens if t not in ["[CLS]", "[SEP]"]]
word_scores = grad_norm[1:len(word_tokens)+1]
axes[0].bar(word_tokens, word_scores, color="#3498db", edgecolor="white")
axes[0].set_title("Integrated Gradients", fontsize=12)
axes[0].set_ylabel("Normalized Score")
plt.setp(axes[0].get_xticklabels(), rotation=30, ha="right")

# LIME
colors = ["#2ecc71" if s >= 0 else "#e74c3c" for s in lime_norm]
axes[1].bar(words, lime_norm, color=colors, edgecolor="white")
axes[1].axhline(0, color="black", linewidth=0.8, linestyle="--")
axes[1].set_title("LIME Attribution", fontsize=12)
axes[1].set_ylabel("Normalized Score")
plt.setp(axes[1].get_xticklabels(), rotation=30, ha="right")

fig.suptitle(f'"{sent}"', fontsize=11, style="italic")
plt.tight_layout()
plt.show()

---
## 3. Layer-wise Probing Classifiers

We train a logistic regression probe on the hidden states of each BERT layer to ask:
> *"How well does layer N encode this linguistic property?"*

In [ ]:
from src.probing_classifiers import HiddenStateExtractor, ProbingExperiment
from src.probing_classifiers import sentiment_probe_data, pos_tag_probe_data

hs_extractor = HiddenStateExtractor("bert-base-uncased")
probe = ProbingExperiment(hs_extractor)

### 3a. Sentiment Probing

In [ ]:
texts, labels = sentiment_probe_data()
sentiment_scores = probe.run(texts, labels, task_name="Sentiment", cv=3)
probe.plot_layer_probing(sentiment_scores, task_name="Sentiment")

### 3b. POS-tag Probing

In [ ]:
texts2, labels2 = pos_tag_probe_data()
pos_scores = probe.run(texts2, labels2, task_name="POS (proxy)", cv=3)
probe.plot_layer_probing(pos_scores, task_name="POS (proxy)")

### 3c. Compare Both Probing Tasks

In [ ]:
layers = sorted(sentiment_scores.keys())
sent_acc = [sentiment_scores[l] for l in layers]
pos_acc  = [pos_scores[l] for l in layers]

fig, ax = plt.subplots(figsize=(11, 5))
ax.plot(layers, sent_acc, marker="o", linewidth=2.5, label="Sentiment", color="#e74c3c")
ax.plot(layers, pos_acc,  marker="s", linewidth=2.5, label="POS (proxy)", color="#3498db")
ax.fill_between(layers, sent_acc, alpha=0.1, color="#e74c3c")
ax.fill_between(layers, pos_acc,  alpha=0.1, color="#3498db")
ax.set_xlabel("Transformer Layer", fontsize=12)
ax.set_ylabel("Probing Accuracy (CV)", fontsize=12)
ax.set_title("Layer-wise Probing: Sentiment vs POS", fontsize=13)
ax.set_xticks(layers)
ax.set_xticklabels([f"L{l}" for l in layers], fontsize=9)
ax.legend(fontsize=11)
ax.grid(axis="y", alpha=0.3)
ax.set_ylim(0, 1.05)
plt.tight_layout()
plt.show()

---
## 4. Summary

| Technique | Key Insight |
|-----------|-------------|
| Attention Heatmaps | Early layers attend locally; later layers attend globally |
| Head Diversity | Each head specializes in a different relationship type |
| Integrated Gradients | Assigns credit based on gradient × embedding magnitude |
| LIME | Model-agnostic; reveals decision boundaries via perturbation |
| Probing | Syntactic features peak in middle layers; semantics in later layers |

---
**Author:** Trinadh Kolluboyina | [GitHub](https://github.com/trinadh3nadh) | [LinkedIn](https://linkedin.com/in/trinadhkolluboyina)